In [18]:
import os
from google.colab import userdata

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

# Retrieve JWT Secret Key and expose it as an environment variable
jwt_secret = userdata.get('JWT_SECRET_KEY')
os.environ['JWT_SECRET_KEY'] = jwt_secret

print("Successfully configured AWS credentials, config, and set JWT_SECRET_KEY")

Successfully configured AWS credentials, config, and set JWT_SECRET_KEY


In [19]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists...
Building dependency tree...
Reading state information...
55 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [20]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.005s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.005s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
->   ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


In [21]:
%%jshell

runPy("""
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(
                f'Unable to read Colab Secret: {name}'
            ) from exc
        return None

    if required and not value:
        raise RuntimeError(
            f'Add the Colab Secret {name} and grant this notebook access.'
        )

    return value


REPO_ROOT = Path('/content/BITS_programming')

if not REPO_ROOT.exists():
    subprocess.run(
        [
            'git',
            'clone',
            'https://github.com/aqwertyuiop48/BITS_programming.git',
            str(REPO_ROOT)
        ],
        check=True,
    )

NOTEBOOK_DIR = (
    REPO_ROOT
    / 'assignments/assignment_15/class3/class3'
)

os.chdir(NOTEBOOK_DIR)

print(
    f'Working directory: {NOTEBOOK_DIR}'
)
""");

->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Working directory: /content/BITS_programming/assignments/assignment_15/class3/class3


# Part 3: Automated Testing

**Class 3 — Non-Functional Testing & AI Security**

---

## Objective

1. Write **pytest scripts** for three test categories:
   - API response tests (correctness & error handling)
   - Accuracy threshold tests (model quality gate)
   - Latency threshold tests (performance SLA)
2. Understand **Continuous Integration (CI)** and how automated tests gate deployment
3. See how an **automated ML testing pipeline** works end-to-end

---

## Key Concepts

| Concept | Definition |
|---------|------------|
| **pytest** | Python test framework — auto-discovers `test_*.py` files and `test_*` functions |
| **Fixture** | Reusable test setup/teardown declared with `@pytest.fixture` |
| **Assertion** | `assert <condition>` — test fails if condition is False |
| **CI/CD** | Continuous Integration/Delivery — run tests on every code push |
| **Quality Gate** | Minimum passing criteria before code can merge to main |
| **SLA** | Service Level Agreement — e.g., p99 latency < 200ms |

---

## The Testing Pyramid for ML Systems

```
        ┌──────────────────┐
        │  End-to-End      │  ← Slow, few: full prediction workflow
        ├──────────────────┤
        │  Integration     │  ← Medium: multi-model, canary, metrics
        ├──────────────────┤
        │  API Response    │  ← Fast, many: endpoint correctness
        ├──────────────────┤
        │  Unit: Accuracy  │  ← Model quality gates
        ├──────────────────┤
        │  Unit: Latency   │  ← Performance SLA gates
        └──────────────────┘
```

Class 3 focuses on the **API Response**, **Accuracy**, and **Latency** layers.

In [22]:
%%jshell

runPy("""
# ── Section 1: Setup & Imports ──────────────────────────────────────────────
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value
import io
import sys
import time
import numpy as np
import pandas as pd
import pytest
from pathlib import Path
from PIL import Image
from datetime import datetime, timedelta
from typing import Optional, Dict
from collections import defaultdict

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from fastapi import FastAPI, File, UploadFile, HTTPException, Depends, Request
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel
import subprocess
subprocess.run(['pip', 'install', 'python-jose', 'bcrypt'], check=True)
from jose import JWTError, jwt
import bcrypt as _bcrypt

# Constants
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['animal', 'name_board', 'vehicle', 'pedestrian',
               'pothole', 'road_sign', 'speed_breaker']
NUM_CLASSES = len(CLASS_NAMES)
DATASET_PATH = str(REPO_ROOT / 'module_4/dataset/dataset')
TEST_PATH = os.path.join(DATASET_PATH, 'test')

# Test thresholds
# NOTE: These use values compatible with RANDOM weights (no training).
# In production with a trained model, set:
#   ACCURACY_THRESHOLD = 70.0  (trained ResNet-18 achieves ~83%)
#   LATENCY_P99_SLA_MS = 100   (tighter SLA for production)
ACCURACY_THRESHOLD  = 10.0   # Random weights → ~14% (chance level)
LATENCY_P99_SLA_MS  = 500    # ms — conservative for CPU demo
SINGLE_LATENCY_MS   = 200    # ms — per-request SLA

print(f'Device: {DEVICE}')
print(f'Test thresholds: accuracy>{ACCURACY_THRESHOLD}%, latency p99<{LATENCY_P99_SLA_MS}ms')
print('All imports successful!')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Requirement already satisfied: python-jose in /usr/local/lib/python3.13/dist-packages (3.5.0)
Device: cpu
Test thresholds: accuracy>10.0%, latency p99<500ms
All imports successful!


In [23]:
%%jshell

runPy("""
import zlib, base64
_c = [
(4027,'eNrFV+Fu20YS/q+nWKDAcXkns7Icp6nuCJxsy7UviWNYcoNCEIgVuZQ3pUiW'),
(1312,'u4ytCgLuIe4J70numyUpUo7zI+2PEwSR4s7Mzs58883wO/bf//wbXzaVoVFZ'),
(1389,'yoYj9j6LZML+wsa313hsyryW+b9/e73vauciGatUWYe5FmvJhGZhIrQeuj17'),
(2713,'ZeOL8dTK8jT1cFMm0h31GD7QZUFA6kHAtUziPkvLdWDVpPZ/qMXoo8tcFtz1'),
(8491,'9uJuuwRFr5A6lYb5rLo5fsMfpVo9GO3fZKl8UdiLQ8jDp3cqlaLgp8fDg/3d'),
(9982,'vY9xVjyKIqpdfHJH2MWURdq1xp/cXs8UItWQXsPw/l5759k6z7Tkc2uxs3An'),
(4900,'tfpdcn48fNNn+HHdfnd5ls1kqjOcu/9c8wa/IiHltRSpPx94r96c9hkup6/t'),
(7086,'ZfB60WfaRLQ0HP5Iz4bDV9XldAGDC/i7tin0Oym6uX8fnL8bT6eTqeuZjF9M'),
(9512,'fr4+n7iVpCc/iwSRR/KPjo7Yvz7OEAFCJY+LbM1uRWHY0KW13nRyfjeZBW8n'),
(3882,'v8D8SpogzBKxDLQMETruQDVoRRy3N37304e769nVewZ552o6PH3t9DJRmodh'),
(717,'oMMHCWD57MOY/t8iO49ZEZ0hacCEyX6V6X2R+I69g63L8dtJcD+d3E2DizOo'),
(3548,'bW3wHCO1KbUsnBHbOnSTAq740y709zDZf5wHoR9kFOT1ppAPlmGxyY1HK/kj'),
(7132,'X1r9/Xp/v75C7kRiuOu+ZDdSWiwTSQYvRaLlrrfr9QhsCJAwMhBhKLUO7Jl4'),
(6409,'JIzoM/mUK4AtQCaMqHBdFYjJApmGyBAOS6JemOWbukIqpWpBGrWWXmnCNHvk'),
(870,'Lvsb4wcmWVYwkrB/+FqlJU7mH5+67uEuXpmTMb51oA7/KyO7SqqujE+Pxquk'),
(1853,'+V6vz9qU95lIVlmhzMPa3+ceyBJ6k4a26CxoyqKQqQkoO1WiR8B0gdNcyFym'),
(4400,'keYHEHGbeBSbljlysUkyERGuyKlI1k7B2Fcc0v5879KipY4GMjBU20SKgWVd'),
(9286,'Lp1WSsWtoNKMsgS6EEpLdjWb3U6eQpkTW3JtBFAXkDf+q8FxH4c2QgHG1ymq'),
(6654,'TEWsgXOVRVKjipsURVa0p/sTll9AZefzIEUkC+1vnY8fPx5R4SERKkTeqWSq'),
(2126,'2nPqnNOBEZWDurOxaSJRiSE0aWas9DeG5J42IN04K9PIOUAamSNGut1Egjxk'),
(3410,'FgtC191nZgvoTGhpCa5GSLe6LKT+XgUlMJtc2ge9Wv+2kJGyDRlknWcpaPy5'),
(9175,'sXwvUpsKszRWEUAPUzGAYqyYtRfkRbYUS5WgZUo9YhdQnEOpXwkurKQl2+Az'),
(8059,'wt/aTBD5NNwEa93YrFn4UmhD84HIc8u8dPWbp9wok0jfIYJnM/AUjRLgqNq2'),
(2284,'75x4A2+AgPb+CT0PXQqA/r7GB/VSe+LAOuTbULqdEk2ylUo5NaSAaGf0jKDv'),
(6506,'5G8ltrys+mFTsU2RfhU0e3veV+FDTEX3DdUi4+Gv4OJWteHjhoSc0sRHbxz0'),
(3948,'V9Kff0HrC/dP1hRoO6IKAZc7DV8iXDjhS4S+tawxqp3ZN6PFrv8CA58M3APE'),
(570,'b52uLVipycxpIUw1uqxrtM6t5arvUdaJeXC6Wawe8XaqgXP2tGSkWtwADE6N'),
(6065,'GXpaoQbPIvlZhbQbMNrMCrtDMNXV8SWcvqysrle1Go9Vgiq6z4lvL3FPkMGF'),
(1264,'e56HZHYbxIiRQgdpzztIgzycriA52w3pp26VIkmyRxnRxOB4n/IVHRBXWd3k'),
(3259,'aXVdqdhel+scsa3ImazdCgSRvPXox8LW02UcqyePzGKIIwQfCDCJxs8cp4E3'),
(9492,'GSJYq7Tx5VtAOdiDMt6jknZiFadtYX1XQxMEBTYxGm6LR6FM5RZwGvGDWmvk'),
(7226,'Rt+0vTNZ52ZjbTalgIbM1FqsKH3XdPUy5IirzDvbAOLXH3izletiekmBNWDn'),
(6241,'7qezw/6335veMuToDwbFOkIBacKB09pnHk3j88GC/YOdDIljOk+Pq6d/LCGO'),
(8782,'PTPqNGMaY3vCqLZh7elk6DYhspN+972B291dr0w1eFRi1B8cjOSk9Yh5BWaL'),
(2096,'8MFLs2BVUAJbF7PSMJp7bL3xaod2UqFORAio1HUWm7V44tCB22rtH7eS1M/6'),
(6622,'TEVPe2mStPp4bank0J+wyjs1xY6qSnPZX9nxYDDoUtgLXbXjV7Pm2zeR4Gb8'),
(6337,'fjKdY3tPGbnmLl5s2g7rFzQScNsSCURxLeT2X3Xm7hd6r789sL6o22p1rPmg'),
(5909,'rxYuvfgxReWInKzkwcvRrjV+0K595/OJ5ca2Ydcu4kF/CCbHYKGAdESLGvK5'),
(1558,'/cNBmPa9CoE5EuFvJc3sVQMhH0Rn/ML4BjXdq8ieOJV6jLXyvH9TJ/S/9q7D'),
(5825,'nM47zeE7DMa68f3sKph9eDu5YTbpzV7eJw2cu/PDFrSo5K8m4ws0ckugNAlg'),
(1083,'nP5dmKpnxPXMyLat6R01p7xQOL9jR5g06sSEER9tPNRHJRJbm3VU6hhFo669'),
(3671,'+ehksNihM0Dnf7DVuzo='),
]
_bad = [i + 1 for i, (k, c) in enumerate(_c) if zlib.crc32(c.encode()) % 10000 != k]
assert not _bad, 'Bad chunk numbers (1-based): ' + str(_bad)
_b = str().join(c for k, c in _c)
exec(compile(zlib.decompress(base64.b64decode(_b)).decode('utf-8'), 'cell', 'exec'), globals())
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... cell:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
API and TestClient ready.
Auth token acquired: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...


## Section 3: Test Helpers

Helper functions used across all test classes.

In [24]:
%%jshell

runPy("""
# ── Section 3: Test Helpers ──────────────────────────────────────────────────

def create_test_image(size=(100, 100), color='red', fmt='PNG'):
    img = Image.new('RGB', size, color=color)
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    buf.seek(0)
    return buf, f'test.{fmt.lower()}'


def post_predict(image_buf, filename='test.png'):
    return client.post(
        '/predict',
        headers=AUTH_HEADERS,
        files={'file': (filename, image_buf, 'image/png')},
    )


def evaluate_model_accuracy(eval_model, loader, device):
    eval_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            _, preds = torch.max(eval_model(images), 1)
            correct += (preds.cpu() == labels).sum().item()
            total   += labels.size(0)
    return round(100 * correct / total, 2) if total > 0 else 0.0


# Test helper functions
buf, fname = create_test_image()
resp = post_predict(buf, fname)
print(f'Helper sanity check: {resp.status_code} → {resp.json().get("prediction", "error")}')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Helper sanity check: 200 → speed_breaker


## Section 4: Category 1 — API Response Tests

These tests verify that the API:
- Returns correct HTTP status codes
- Includes required fields in responses
- Handles errors gracefully
- Enforces security (401 without token)

**pytest convention**: Test methods start with `test_`, organized in classes starting with `Test`.

In [26]:
%%jshell

runPy("""
import zlib, base64
_c = [
(9222,'eNrFWN1u2zYUvvdTHLQXlAHbidOgF0Y9wO0CpEC7BU17lRkCLVE2V4nURCqJ'),
(8333,'FwTYQ+wJ+yQ9/LElS7bjrZknCEhEHZ5zvo8fz6H8Er79/RfecM0izaWA8xFM'),
(8898,'rt7DJ6ZyKRSDz0xp5Y3+37vTiVKqlM0IU1xlOOp0AK+X0O/DJaOpXgATcS65'),
(5708,'0DjkXsYsAY3TwoU1CFcGYcF0WQgVnp2eBoqlSXdk7c1VoH8YQ5RyJvRgznRA'),
(6352,'Ttx00l0bYT6s0NZ2oDTVpQojGTMYjwFd9iAhF/c5Msti9zyXGh6a1o9ke5aF'),
(56,'RxguqAq9fcJZGv9IqsQ5IsCFS/t3JUXQ7QH5yJXiYg7OAGwkshWpm3KzcjU1'),
(1238,'cIkLuDwEyy0rFGrtGcB4T3vQeIsVnEorVwWLudN8pRdQZRQxFFlEFVNt/eRu'),
(4646,'Ul03YSKL8JamPA55RuesiWdWJqgDQTNmUBWMahZaX8662wSeyypOUE1+bs2N'),
(8730,'/Jhm97olwAqmXzUuorSMmVq92bl6B6ONqaZ70PqFbC14Fd+uufGCi72uVplf'),
(1995,'9cpsh47NxJu6t6lx9+7D5Po6/GXy8eK6B7+t55grIV/EVyHvRM03cmj9vKiG'),
(9785,'Xkz/AZmRFAmPmYjYsWmsIq9pbNqcDk7hzdgTVbOfmtHh4LRN0Lu10YqXahry'),
(225,'ArLUIBMoqJgzuEEHw+kurmylxyc5ozOecs2RLVVmoZahFMdnq51NmzYtNU3R'),
(765,'OaYZeNK2TJsOsFSUTAXdVhg6U4Fz0jf8duGNWYNhm+erukcTD2PDg506Gpwn'),
(4875,'jz1gqwKAfsgTJYymqafbcKCOxS1WTazxqrHrRhtQPTHebA+pTY68Hu1x4QGn'),
(1204,'P64LQ1LIDDZn7+BnvUtThCyiZZipY+uuirxzlzpSaoZT+Alwa5EPbgiyUmmY'),
(3380,'MRMb4d6yegu8KApchQUVcWq4aXU7LlxfS3iKwJY5W4vmvH1e0vfaGiJOLgdv'),
(7519,'l+jh/a/BjOgFx+VTILANUQGWGtLd0e4NQ8HGUpITTxjpAR4DYmzn48mXz5fh'),
(8145,'5cXk54tP173NzYEJqPEDMX/JCAISy6jMjGdMDz2skkSCTOM7yVPKBek+Vl4O'),
(5681,'6rPnjT57fvjZLpJFUeZeIvv49Ibb6OzjjXT2ad+uTz8X877j5njE+vQGGBsd'),
(8852,'+Cek1eI6MaP/htUGWVrKUGWmQj1J1xN7UfE/2TgYvsam87rbNfp/y1J5B6/O'),
(9569,'7l+dYXUQPCuz5zyKbZGIrXmai6XbBIcqhmW5Xro9uAe/tWqK5XhysOG9GOz/'),
(2605,'zyQFW6fw27QsuF6OgJZ6gfnziNqzXatilaJmwWIk7I/SjFfEDX/0fF5nsE5a'),
(5076,'kxLrsufcb1BxIAFbGn9NTEO443phTlRafmXiUC3dFVLMQ0QZG45oqvYxsxWw'),
(4652,'DYdwTeNBtCUiMEARMTERzDO+JTliu5NFbMZtUHIw7kbKrgW1Uw5tJv9VzmZ8'),
(768,'/fx46MdXs+HXPklbzZ3aD02H4on2bm1sA/bf2jNGC8y50+nkBX63BqTxmwi4'),
(6918,'X0qQQy5QLkYqWPcsn2qAReE7rMrJFg=='),
]
_bad = [i + 1 for i, (k, c) in enumerate(_c) if zlib.crc32(c.encode()) % 10000 != k]
assert not _bad, 'Bad chunk numbers (1-based): ' + str(_bad)
_b = str().join(c for k, c in _c)
exec(compile(zlib.decompress(base64.b64decode(_b)).decode('utf-8'), 'cell', 'exec'), globals())
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... TestAPIResponse class defined with 16 tests.


## Section 5: Category 2 — Accuracy Threshold Tests

These tests verify that the model meets a **minimum accuracy requirement** before deployment.

This is a **quality gate** — if accuracy drops below the threshold, the CI pipeline fails and the code cannot merge.

> **Note on thresholds**: We use `ACCURACY_THRESHOLD = 10%` here because we're using random weights (chance level ≈ 14.3% for 7 classes). In production with a **trained model** (ResNet-18 achieves ~83%), set `ACCURACY_THRESHOLD = 70.0`.

In [27]:
%%jshell

runPy("""
# ── Section 5: Accuracy Threshold Tests ─────────────────────────────────────

class TestAccuracyThreshold:

    # Shared test dataset (loaded once)
    _test_loader = None

    @classmethod
    def get_test_loader(cls):
        if cls._test_loader is None:
            test_ds = ImageFolder(TEST_PATH, transform=transform)
            cls._test_loader = DataLoader(
                test_ds, batch_size=32, shuffle=False, num_workers=0
            )
        return cls._test_loader

    def test_overall_accuracy_above_threshold(self):
        loader = self.get_test_loader()
        accuracy = evaluate_model_accuracy(model, loader, DEVICE)
        print(f'  Overall accuracy: {accuracy:.2f}% (threshold: {ACCURACY_THRESHOLD}%)')
        assert accuracy >= ACCURACY_THRESHOLD, \
            f'Accuracy {accuracy:.2f}% below threshold {ACCURACY_THRESHOLD}%'

    def test_per_class_accuracy_not_zero(self):
        loader = self.get_test_loader()
        predicted_classes = set()
        model.eval()
        with torch.no_grad():
            for images, _ in loader:
                _, preds = torch.max(model(images.to(DEVICE)), 1)
                predicted_classes.update(preds.cpu().numpy().tolist())
        unique_preds = len(predicted_classes)
        print(f'  Unique classes predicted: {unique_preds}/{NUM_CLASSES}')
        assert unique_preds > 1, \
            f'Model only predicts {unique_preds} class(es) — possible degenerate model'

    def test_confidence_values_in_valid_range(self):
        for color in ['red', 'green', 'blue', 'white']:
            buf, fname = create_test_image(color=color)
            data = post_predict(buf, fname).json()
            assert 0.0 <= data['confidence'] <= 1.0, \
                f'Confidence {data["confidence"]} out of range'
            total = sum(data['class_probabilities'].values())
            assert abs(total - 1.0) < 0.01, \
                f'Probabilities sum to {total:.4f} for {color} image'

    def test_model_prediction_deterministic(self):
        predictions = []
        for _ in range(5):
            buf, fname = create_test_image(color='purple', size=(128, 128))
            data = post_predict(buf, fname).json()
            predictions.append(data['prediction'])
        assert len(set(predictions)) == 1, \
            f'Non-deterministic predictions: {predictions}'

    def test_prediction_is_valid_class(self):
        for _ in range(3):
            buf, fname = create_test_image()
            data = post_predict(buf, fname).json()
            assert data['prediction'] in CLASS_NAMES, \
                f'Unknown prediction class: {data["prediction"]}'


print('TestAccuracyThreshold class defined with 5 tests.')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... TestAccuracyThreshold class defined with 5 tests.


## Section 6: Category 3 — Latency Threshold Tests

These tests verify that the API meets **performance SLA** requirements.

**Why test latency?** An ADAS car traveling at 60 km/h moves 16 meters/second. A 500ms delay means the car has moved 8 meters without updated hazard detection.

**SLA values used here** (for CPU demo with random weights):
- Single request: < 200ms
- p99 across 50 requests: < 500ms

In production: tighten to <50ms single, <100ms p99.

In [28]:
%%jshell

runPy("""
# ── Section 6: Latency Threshold Tests ──────────────────────────────────────

class TestLatencyThreshold:

    def test_single_image_latency_under_sla(self):
        buf, fname = create_test_image(size=(128, 128))
        start   = time.perf_counter()
        resp    = post_predict(buf, fname)
        elapsed = (time.perf_counter() - start) * 1000
        assert resp.status_code == 200
        print(f'  Single request latency: {elapsed:.2f}ms (SLA: <{SINGLE_LATENCY_MS}ms)')
        assert elapsed < SINGLE_LATENCY_MS, \
            f'Latency {elapsed:.2f}ms exceeds SLA of {SINGLE_LATENCY_MS}ms'

    def test_p99_latency_under_sla(self):
        n_requests = 50
        latencies  = []
        for _ in range(n_requests):
            buf, fname = create_test_image()
            start   = time.perf_counter()
            resp    = post_predict(buf, fname)
            elapsed = (time.perf_counter() - start) * 1000
            assert resp.status_code == 200
            latencies.append(elapsed)

        p50 = np.percentile(latencies, 50)
        p95 = np.percentile(latencies, 95)
        p99 = np.percentile(latencies, 99)
        print(f'  p50={p50:.1f}ms  p95={p95:.1f}ms  p99={p99:.1f}ms  '
              f'(SLA: p99<{LATENCY_P99_SLA_MS}ms)')
        assert p99 < LATENCY_P99_SLA_MS, \
            f'p99 latency {p99:.2f}ms exceeds SLA of {LATENCY_P99_SLA_MS}ms'

    def test_latency_reported_by_api_is_positive(self):
        buf, fname = create_test_image()
        data = post_predict(buf, fname).json()
        assert data['latency_ms'] > 0, 'API-reported latency must be positive'

    def test_large_image_within_sla(self):
        buf, fname = create_test_image(size=(640, 480))
        start   = time.perf_counter()
        resp    = post_predict(buf, fname)
        elapsed = (time.perf_counter() - start) * 1000
        assert resp.status_code == 200
        sla_large = SINGLE_LATENCY_MS * 2
        print(f'  640x480 image latency: {elapsed:.2f}ms (SLA: <{sla_large}ms)')
        assert elapsed < sla_large, \
            f'Large image latency {elapsed:.2f}ms exceeds 2x SLA of {sla_large}ms'

    def test_successive_requests_consistent_latency(self):
        latencies = []
        for _ in range(20):
            buf, fname = create_test_image()
            start = time.perf_counter()
            post_predict(buf, fname)
            latencies.append((time.perf_counter() - start) * 1000)

        first_half_mean = np.mean(latencies[:10])
        second_half_mean = np.mean(latencies[10:])
        degradation_pct = ((second_half_mean - first_half_mean) / first_half_mean) * 100
        print(f'  Latency: first 10 mean={first_half_mean:.1f}ms, '
              f'last 10 mean={second_half_mean:.1f}ms, '
              f'degradation={degradation_pct:.1f}%')
        # Allow up to 50% degradation (warming up effects on cold start)
        assert degradation_pct < 50, \
            f'Latency degraded by {degradation_pct:.1f}% over 20 requests (possible memory leak)'


print('TestLatencyThreshold class defined with 5 tests.')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... TestLatencyThreshold class defined with 5 tests.


## Section 7: Run All Tests

We run all three test categories and print a summary report.

In [29]:
%%jshell

runPy("""
# ── Section 7: Test Runner ───────────────────────────────────────────────────

def run_test_class(test_class):
    instance = test_class()
    results  = []
    methods  = [m for m in dir(instance) if m.startswith('test_')]

    for method_name in sorted(methods):
        method = getattr(instance, method_name)
        try:
            method()
            results.append((method_name, 'PASS', None))
        except AssertionError as e:
            results.append((method_name, 'FAIL', str(e)))
        except Exception as e:
            results.append((method_name, 'ERROR', str(e)))

    passed = sum(1 for _, s, _ in results if s == 'PASS')
    failed = len(results) - passed
    return passed, failed, results


def print_test_results(class_name, passed, failed, results):
    total = passed + failed
    status = '✓ ALL PASS' if failed == 0 else f'✗ {failed} FAILED'
    print(f'\\n  {class_name}: {passed}/{total} passed  [{status}]')
    for name, state, err in results:
        icon = '  ✓' if state == 'PASS' else '  ✗'
        print(f'{icon} {name}')
        if err: print(f'     → {err[:80]}')


# Run all test categories
print('=' * 65)
print('  Automated Test Suite — Class 3 Part 3')
print('=' * 65)

total_pass = total_fail = 0

for cls in [TestAPIResponse, TestAccuracyThreshold, TestLatencyThreshold]:
    p, f, r = run_test_class(cls)
    print_test_results(cls.__name__, p, f, r)
    total_pass += p
    total_fail += f

print('\\n' + '=' * 65)
print(f'  TOTAL: {total_pass} passed, {total_fail} failed')
overall = 'ALL TESTS PASSED ✓' if total_fail == 0 else f'{total_fail} TESTS FAILED ✗'
print(f'  {overall}')
print('=' * 65)

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... =================================================================
  Automated Test Suite — Class 3 Part 3

  TestAPIResponse: 16/16 passed  [✓ ALL PASS]
  ✓ test_corrupt_image_returns_400
  ✓ test_empty_file_returns_400
  ✓ test_health_endpoint_returns_200
  ✓ test_health_response_has_status_field
  ✓ test_health_response_has_version_field
  ✓ test_invalid_file_type_returns_400
  ✓ test_predict_class_probabilities_sum_to_one
  ✓ test_predict_includes_latency_ms
  ✓ test_predict_response_includes_confidence
  ✓ test_predict_response_includes_prediction_field
  ✓ test_predict_returns_200_for_valid_image
  ✓ test_predict_returns_all_class_names
  ✓ test_too_small_image_returns_400
  ✓ test_unauthenticated_request_returns_401
  ✓ test_valid_credentials_return_token
  ✓ test_wrong_credentials_returns_401
  Overall accuracy: 0.85% (threshold: 10.0%)
  Unique classes predicted: 3/7

  TestAccuracyThreshold: 4/5 passed  [✗ 1 F

## Summary & Key Takeaways

### What We Built

| Test Category | Tests | Purpose |
|---------------|-------|--------|
| **API Response** | 16 | Correctness, error handling, auth (401/400) |
| **Accuracy Threshold** | 5 | Model quality gate — block deployment if accuracy drops |
| **Latency Threshold** | 5 | SLA gate — block deployment if latency exceeds limit |
| **Total** | **26** | Comprehensive non-functional test coverage |

### pytest Best Practices

1. **Test one thing per function** — `test_health_returns_200` not `test_health`
2. **Descriptive names** — name says what it tests AND what the expected behavior is
3. **Use fixtures** — `@pytest.fixture` for reusable setup (test images, tokens, DB state)
4. **Arrange-Act-Assert (AAA)** — setup → call → verify
5. **Test edge cases** — empty files, corrupt images, missing fields, boundary values

### CI/CD for ML Systems

A mature ML CI pipeline includes:
- **Functional tests**: Does the API work correctly?
- **Non-functional tests**: Is it fast enough? Accurate enough? Secure?
- **Data validation**: Is the training data within expected distribution?
- **Model regression**: Did the new model do WORSE than the current production model?
- **Security scan**: Does the code have known vulnerabilities? (bandit, safety)

### Next Step: Lab 3

Now it's your turn. Lab 3 combines everything from Parts 1–3 into a **Production-Grade Testing Checklist**.

In [30]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 15:47:31
